# 02 — Discrete vs continuous dividends

Same option priced with `dividend_model="discrete"` ($S^* = S_0 - \mathrm{PV(div)}$) and `"continuous"` (equivalent yield). Q/A: `docs/chapters/equity.tex` (Forwards and dividends).

In [ ]:
import os, sys
from datetime import timedelta
from math import erf, exp, log, sqrt
from pathlib import Path
os.environ["TF_ENABLE_ONEDNN_OPTS"] = "0"

REPO = Path.cwd().resolve().parents[1]
sys.path.insert(0, str(REPO / "examples"))

import numpy as np
import matplotlib.pyplot as plt
import tensorquant as tq
from market_data import EVALUATION_DATE, build_market_environment

## Market

In [ ]:
tq.Settings.evaluation_date = EVALUATION_DATE
mkt = build_market_environment(EVALUATION_DATE)

TICKER = "TICKER1"
ccy    = mkt.get_currency(TICKER)
S0     = float(mkt.get_eq_spot(TICKER, ccy=ccy))
curve  = mkt.get_ir_curve(ccy)
divs   = mkt.get_eq_dividends(TICKER, ccy=ccy)
expiry = EVALUATION_DATE + timedelta(days=365)

T     = (expiry - EVALUATION_DATE).days / 365
pv    = float(divs.pv_dividends(expiry, curve))
q_eff = -log(1 - pv / S0) / T

for ex, d in divs.dividends_before(expiry):
    print(f"ex {ex}  D = {d:.2f}")
print(f"S0 = {S0:.2f}  PV(div) = {pv:.4f}  q_eff = {q_eff:.4%}")

## Forward
$F_{disc} = (S_0 - PV)\,e^{(r-q_{repo})T}$, $\quad F_{cont} = S_0\,e^{(r-q_{eff}-q_{repo})T}$ with $q_{eff} = -\ln(1-PV/S_0)/T$ $\Rightarrow$ same forward.

In [ ]:
MODELS = ["discrete", "continuous"]

def price(model, K, opt_type=tq.OptionType.Call, exercise=tq.ExerciseType.European, autodiff=False):
    opt = tq.VanillaOption(ccy, start_date=EVALUATION_DATE, end_date=expiry, option_type=opt_type,
                           strike=K, underlying=TICKER, exercise_type=exercise)
    pricer = tq.BlackScholesPricer(dividend_model=model, use_implied_repo=True)
    pricer.price(opt, mkt, autodiff=autodiff)
    return opt, pricer

for m in MODELS:
    o, _ = price(m, S0)
    print(f"{m:<10s}  S_net = {float(o.spot_net):8.4f}  F = {float(o.forward):8.4f}  "
          f"price = {float(o.price):.4f}")

## European prices: identical
Black–Scholes depends only on $(F, \sigma, T, DF)$.

In [ ]:
strikes = np.arange(80.0, 121.0, 5.0)
print(f"{'K':>6s}  {'call disc':>10s}  {'call cont':>10s}  {'put disc':>10s}  {'put cont':>10s}")
for K in strikes:
    c = [float(price(m, K)[0].price) for m in MODELS]
    p = [float(price(m, K, tq.OptionType.Put)[0].price) for m in MODELS]
    print(f"{K:6.1f}  {c[0]:10.4f}  {c[1]:10.4f}  {p[0]:10.4f}  {p[1]:10.4f}")

## Delta: sticky cash vs sticky yield
Discrete: dividends are fixed cash, $\partial F/\partial S = e^{(r-q_{repo})T}$. Continuous: the yield is fixed, dividends scale with $S$ $\Rightarrow$ $\Delta_{cont} \approx \Delta_{disc}\,(S_0-PV)/S_0$.

In [ ]:
def delta(model, K):
    o, pricer = price(model, K, autodiff=True)
    return float(pricer.tape.gradient(o.price, o.spot))

d = {m: np.array([delta(m, K) for K in strikes]) for m in MODELS}
print(f"ratio cont/disc = {np.mean(d['continuous'] / d['discrete']):.4f}   (S0-PV)/S0 = {1 - pv / S0:.4f}")

plt.figure(figsize=(6, 4))
for m in MODELS:
    plt.plot(strikes, d[m], "o-", label=m)
plt.title("Call delta (autodiff)"); plt.xlabel("K"); plt.grid(alpha=0.3); plt.legend()

## American options: early exercise
The dividend model drives the carry $b = r - q$, hence the early-exercise premium.

In [ ]:
for opt_type in (tq.OptionType.Call, tq.OptionType.Put):
    for m in MODELS:
        eu = float(price(m, S0, opt_type)[0].price)
        am = float(price(m, S0, opt_type, tq.ExerciseType.American)[0].price)
        print(f"{opt_type.name:<4s} {m:<10s}  EU = {eu:.4f}  AM = {am:.4f}  EEP = {am - eu:.4f}")

## True cash-dividend dynamics (MC)
GBM on the whole $S$, dropping by $D_i$ at each ex-date. With the same $\sigma$ the price is higher than the escrowed model, where $\sigma$ acts only on $S_0 - PV$. Rule of thumb: $\sigma^* \approx \sigma\,S_0/(S_0 - PV)$.

In [ ]:
o, _ = price("discrete", S0)
sigma, r, repo = float(o.volatility), float(o.risk_free_rate), float(o.repo_margin)
F, DF = float(o.forward), float(o.discount_factor)

# exact GBM between ex-dates, cash drop at each ex-date (flat r)
rng = np.random.default_rng(42)
n = 400_000
t_ex = [((ex - EVALUATION_DATE).days / 365, D) for ex, D in divs.dividends_before(expiry)]
S, t_prev = np.full(n, S0), 0.0
for t, D in t_ex + [(T, 0.0)]:
    dt = t - t_prev
    S = S * np.exp((r - repo - 0.5 * sigma**2) * dt + sigma * sqrt(dt) * rng.standard_normal(n))
    S, t_prev = np.maximum(S - D, 0.0), t
payoff = np.exp(-r * T) * np.maximum(S - S0, 0.0)

def black_call(F, K, sigma, T, DF):
    N = lambda x: 0.5 * (1 + erf(x / sqrt(2)))
    d1 = (log(F / K) + 0.5 * sigma**2 * T) / (sigma * sqrt(T))
    return DF * (F * N(d1) - K * N(d1 - sigma * sqrt(T)))

sigma_star = sigma * S0 / (S0 - pv)
print(f"escrowed (tQuant)     : {float(o.price):.4f}   sigma  = {sigma:.4%}")
print(f"MC cash drops         : {payoff.mean():.4f} +/- {payoff.std() / sqrt(n):.4f}")
print(f"escrowed with sigma*  : {black_call(F, S0, sigma_star, T, DF):.4f}   sigma* = {sigma_star:.4%}")

## Exercises
1. Set `expiry` before the first ex-date: what happens to forward, delta and EEP?
2. Why is the American call in the discrete model worth the same as the European? (hint: carry $b = r - q_{repo}$).
3. Double the dividends in `market_data.py`: how does the MC vs $\sigma^*$ gap evolve?